# 01 - Exploratory Data Analysis & Feature Engineering

## Insurance Claims Prediction ML Project

### Project Overview
This notebook performs exploratory data analysis (EDA) and feature engineering on an insurance claims dataset. The goal is to build a binary classification model that predicts whether a policyholder will file an insurance claim (`claim_filed = 1`) or not (`claim_filed = 0`).

### Dataset Description
The dataset is based on the **Kaggle Insurance Claims Dataset** and contains policyholder demographics, vehicle information, and historical claim data. Key features include:

| Feature | Description |
|---------|-------------|
| `age` | Age of the policyholder |
| `gender` | Gender (Male/Female) |
| `vehicle_age` | Age of the insured vehicle |
| `annual_premium` | Annual premium amount |
| `policy_tenure` | Years the policy has been held |
| `num_claims_hist` | Number of historical claims |
| `credit_score` | Policyholder credit score |
| `region` | Geographic region |
| `vehicle_type` | Type of vehicle |
| `claim_filed` | **Target** - whether a claim was filed (0/1) |

### Notebook Outline
1. Import libraries
2. Load data (with synthetic fallback)
3. Basic dataset information
4. Target distribution analysis
5. Numeric feature distributions
6. Correlation analysis
7. Feature vs. target analysis
8. Categorical feature analysis
9. Feature engineering
10. Summary of findings

## 1. Import Libraries

In [ ]:
import sys
import os
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split

# Add project root to path for src imports
sys.path.insert(0, '..')
from src.data_pipeline import clean_data, engineer_features

# Plot styling
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3
sns.set_style('whitegrid')
sns.set_palette('husl')

print(f"NumPy: {np.__version__}")
print(f"Pandas: {pd.__version__}")
print("Libraries loaded successfully.")

## 2. Load Data

We attempt to load the dataset from the `data/` directory. If no CSV file is found (e.g., the Kaggle dataset hasn't been downloaded), we generate a **synthetic insurance claims dataset** with realistic distributions so the notebook remains fully executable.

In [ ]:
def generate_synthetic_insurance_data(n_samples=10000, random_state=42):
    """
    Generate a synthetic insurance claims dataset with realistic distributions.
    Used as a fallback when the real Kaggle dataset is not available.
    """
    rng = np.random.RandomState(random_state)
    
    # Policyholder demographics
    age = rng.normal(loc=40, scale=12, size=n_samples).clip(18, 75).astype(int)
    gender = rng.choice(['Male', 'Female'], size=n_samples, p=[0.55, 0.45])
    
    # Vehicle information
    vehicle_age = rng.choice(
        ['< 1 Year', '1-2 Year', '> 2 Years'],
        size=n_samples,
        p=[0.25, 0.40, 0.35]
    )
    vehicle_type = rng.choice(
        ['Sedan', 'SUV', 'Hatchback', 'Truck', 'Van'],
        size=n_samples,
        p=[0.35, 0.25, 0.20, 0.12, 0.08]
    )
    
    # Policy details
    annual_premium = rng.lognormal(mean=9.5, sigma=0.6, size=n_samples).clip(5000, 80000).astype(int)
    policy_tenure = rng.exponential(scale=5, size=n_samples).clip(0.5, 30).round(1)
    
    # Risk indicators
    num_claims_hist = rng.poisson(lam=0.8, size=n_samples)
    credit_score = rng.normal(loc=680, scale=80, size=n_samples).clip(300, 850).astype(int)
    
    # Region
    region = rng.choice(
        ['North', 'South', 'East', 'West'],
        size=n_samples,
        p=[0.30, 0.25, 0.20, 0.25]
    )
    
    # Generate target: claim_filed (influenced by features to create realistic correlations)
    # Higher claim probability for: younger drivers, older vehicles, more past claims, lower credit
    claim_prob = (
        0.05
        + 0.15 * (age < 30).astype(float)
        + 0.10 * (vehicle_age == '> 2 Years').astype(float)
        + 0.08 * (num_claims_hist >= 2).astype(float)
        + 0.10 * (credit_score < 600).astype(float)
        + 0.05 * (annual_premium > 25000).astype(float)
        + rng.normal(0, 0.05, n_samples)  # noise
    ).clip(0.01, 0.95)
    claim_filed = rng.binomial(1, claim_prob)
    
    df = pd.DataFrame({
        'age': age,
        'gender': gender,
        'vehicle_age': vehicle_age,
        'annual_premium': annual_premium,
        'policy_tenure': policy_tenure,
        'num_claims_hist': num_claims_hist,
        'credit_score': credit_score,
        'region': region,
        'vehicle_type': vehicle_type,
        'claim_filed': claim_filed
    })
    
    return df


# Try loading from data/ directory, fall back to synthetic data
data_dir = os.path.join('..', 'data')
csv_files = [f for f in os.listdir(data_dir) if f.endswith('.csv')] if os.path.exists(data_dir) else []

if csv_files:
    filepath = os.path.join(data_dir, csv_files[0])
    df = pd.read_csv(filepath)
    print(f"Loaded real dataset from: {filepath}")
    print(f"Shape: {df.shape}")
else:
    print("No CSV files found in data/ directory.")
    print("Generating synthetic insurance claims dataset...")
    df = generate_synthetic_insurance_data(n_samples=10000)
    print(f"Generated synthetic dataset with shape: {df.shape}")

print(f"\nColumns: {df.columns.tolist()}")

## 3. Basic Dataset Information

Let's inspect the structure, data types, summary statistics, and missing values.

In [ ]:
# Shape
print(f"Dataset shape: {df.shape[0]} rows x {df.shape[1]} columns")
print(f"Memory usage: {df.memory_usage(deep=True).sum() / 1024:.1f} KB")
print()

# Data types
print("Data Types:")
print(df.dtypes)
print()

# Info
df.info()

In [ ]:
# Summary statistics for numeric columns
print("Numeric Feature Statistics:")
df.describe().round(2)

In [ ]:
# Summary statistics for categorical columns
print("Categorical Feature Summary:")
df.describe(include='object')

In [ ]:
# Missing values analysis
missing = df.isnull().sum()
missing_pct = (df.isnull().sum() / len(df) * 100).round(2)
missing_df = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})
missing_df = missing_df[missing_df['Missing Count'] > 0].sort_values('Missing %', ascending=False)

if len(missing_df) == 0:
    print("No missing values found in the dataset.")
else:
    print("Missing Values:")
    display(missing_df)

# Check for duplicates
n_duplicates = df.duplicated().sum()
print(f"\nDuplicate rows: {n_duplicates} ({n_duplicates/len(df)*100:.2f}%)")

## 4. Target Distribution

Examining the distribution of `claim_filed` to understand class balance. Insurance claims datasets are typically imbalanced with fewer positive cases.

In [ ]:
# Identify target column
target_col = 'claim_filed'
if target_col not in df.columns:
    # Fall back to any column with 'claim' in the name
    target_candidates = [c for c in df.columns if 'claim' in c.lower() and df[c].nunique() <= 5]
    target_col = target_candidates[0] if target_candidates else df.columns[-1]
    print(f"Using '{target_col}' as target column")

target_counts = df[target_col].value_counts()
target_pct = df[target_col].value_counts(normalize=True) * 100

print(f"Target Distribution ({target_col}):")
print(f"  No Claim (0): {target_counts.get(0, 0):,} ({target_pct.get(0, 0):.1f}%)")
print(f"  Claim    (1): {target_counts.get(1, 0):,} ({target_pct.get(1, 0):.1f}%)")
print(f"  Imbalance Ratio: {target_counts.get(0, 1) / max(target_counts.get(1, 1), 1):.1f}:1")

# Bar chart
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Count plot
colors = ['#3498db', '#e74c3c']
bars = axes[0].bar(
    ['No Claim (0)', 'Claim (1)'],
    [target_counts.get(0, 0), target_counts.get(1, 0)],
    color=colors,
    edgecolor='black',
    alpha=0.85
)
for bar, count in zip(bars, [target_counts.get(0, 0), target_counts.get(1, 0)]):
    axes[0].text(bar.get_x() + bar.get_width()/2., bar.get_height() + 50,
                 f'{count:,}', ha='center', va='bottom', fontweight='bold', fontsize=13)
axes[0].set_title('Target Distribution (Counts)', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Count')

# Pie chart
axes[1].pie(
    [target_counts.get(0, 0), target_counts.get(1, 0)],
    labels=['No Claim (0)', 'Claim (1)'],
    autopct='%1.1f%%',
    colors=colors,
    startangle=90,
    explode=(0, 0.05),
    textprops={'fontsize': 12}
)
axes[1].set_title('Target Distribution (Proportions)', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

## 5. Numeric Feature Distributions

Visualizing the distribution of each numeric feature to identify skewness, outliers, and unusual patterns.

In [ ]:
# Identify numeric columns (exclude target)
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
numeric_cols = [c for c in numeric_cols if c != target_col]

print(f"Numeric features ({len(numeric_cols)}): {numeric_cols}")

# Histograms with KDE
n_cols = 3
n_rows = (len(numeric_cols) + n_cols - 1) // n_cols
fig, axes = plt.subplots(n_rows, n_cols, figsize=(6 * n_cols, 5 * n_rows))
axes = axes.flatten() if n_rows > 1 else [axes] if len(numeric_cols) == 1 else axes.flatten()

for i, col in enumerate(numeric_cols):
    ax = axes[i]
    
    # Histogram with KDE
    ax.hist(df[col].dropna(), bins=40, alpha=0.7, color='#3498db', edgecolor='black', density=True)
    df[col].dropna().plot.kde(ax=ax, color='#e74c3c', linewidth=2)
    
    # Stats annotation
    mean_val = df[col].mean()
    median_val = df[col].median()
    skew_val = df[col].skew()
    ax.axvline(mean_val, color='red', linestyle='--', alpha=0.7, label=f'Mean: {mean_val:.1f}')
    ax.axvline(median_val, color='green', linestyle='--', alpha=0.7, label=f'Median: {median_val:.1f}')
    
    ax.set_title(f'{col} (skew: {skew_val:.2f})', fontsize=13, fontweight='bold')
    ax.legend(fontsize=9)

# Hide unused axes
for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Numeric Feature Distributions', fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Box plots to identify outliers
fig, axes = plt.subplots(1, len(numeric_cols), figsize=(4 * len(numeric_cols), 6))
if len(numeric_cols) == 1:
    axes = [axes]

for i, col in enumerate(numeric_cols):
    bp = axes[i].boxplot(df[col].dropna(), patch_artist=True, vert=True)
    bp['boxes'][0].set_facecolor('#3498db')
    bp['boxes'][0].set_alpha(0.7)
    axes[i].set_title(col, fontsize=12, fontweight='bold')
    
    # Count outliers using IQR method
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    n_outliers = ((df[col] < Q1 - 1.5 * IQR) | (df[col] > Q3 + 1.5 * IQR)).sum()
    axes[i].set_xlabel(f'Outliers: {n_outliers}', fontsize=10)

plt.suptitle('Outlier Detection (Box Plots)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 6. Correlation Heatmap

Examining pairwise correlations between numeric features and the target variable. High correlations between features may indicate multicollinearity, while correlations with the target reveal predictive features.

In [ ]:
# Compute correlation matrix (numeric features + target)
corr_cols = numeric_cols + [target_col]
corr_matrix = df[corr_cols].corr()

# Full heatmap
fig, ax = plt.subplots(figsize=(12, 10))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool), k=1)
sns.heatmap(
    corr_matrix,
    mask=mask,
    annot=True,
    fmt='.3f',
    cmap='RdBu_r',
    center=0,
    vmin=-1,
    vmax=1,
    square=True,
    linewidths=0.5,
    ax=ax,
    cbar_kws={'label': 'Pearson Correlation'}
)
ax.set_title('Feature Correlation Heatmap', fontsize=16, fontweight='bold', pad=20)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

# Print correlations with target
print(f"\nCorrelations with '{target_col}' (sorted by absolute value):")
target_corr = corr_matrix[target_col].drop(target_col).abs().sort_values(ascending=False)
for feat, corr in target_corr.items():
    direction = '+' if corr_matrix.loc[feat, target_col] > 0 else '-'
    print(f"  {feat:25s} {direction}{corr:.4f}")

## 7. Feature vs. Target Analysis

Analyzing how each numeric feature differs between policyholders who filed a claim vs. those who didn't.

In [ ]:
# Box plots: each numeric feature split by claim_filed
n_cols_plot = 3
n_rows_plot = (len(numeric_cols) + n_cols_plot - 1) // n_cols_plot
fig, axes = plt.subplots(n_rows_plot, n_cols_plot, figsize=(6 * n_cols_plot, 5 * n_rows_plot))
axes = axes.flatten()

for i, col in enumerate(numeric_cols):
    sns.boxplot(
        data=df, x=target_col, y=col, ax=axes[i],
        palette={0: '#3498db', 1: '#e74c3c'},
        width=0.5
    )
    axes[i].set_title(f'{col} by {target_col}', fontsize=13, fontweight='bold')
    axes[i].set_xlabel('Claim Filed')
    
    # Add mean annotations
    means = df.groupby(target_col)[col].mean()
    for cls_val, mean_val in means.items():
        axes[i].annotate(
            f'\u03bc={mean_val:.1f}',
            xy=(cls_val, mean_val),
            fontsize=9, fontweight='bold', color='black',
            ha='center', va='bottom'
        )

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Numeric Features by Claim Status', fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Statistical comparison: mean values by claim status
comparison = df.groupby(target_col)[numeric_cols].agg(['mean', 'median', 'std']).round(2)
print("Feature Statistics by Claim Status:")
comparison

## 8. Categorical Feature Analysis

Examining the distribution of categorical features and their relationship with the target variable.

In [ ]:
# Identify categorical columns
categorical_cols = df.select_dtypes(include=['object', 'category']).columns.tolist()
print(f"Categorical features ({len(categorical_cols)}): {categorical_cols}")

# Count plots for each categorical feature
n_cat = len(categorical_cols)
if n_cat > 0:
    fig, axes = plt.subplots(1, n_cat, figsize=(6 * n_cat, 5))
    if n_cat == 1:
        axes = [axes]
    
    for i, col in enumerate(categorical_cols):
        order = df[col].value_counts().index
        sns.countplot(data=df, x=col, ax=axes[i], order=order, palette='husl', edgecolor='black')
        axes[i].set_title(f'Distribution of {col}', fontsize=13, fontweight='bold')
        axes[i].tick_params(axis='x', rotation=45)
        
        # Annotate counts
        for p in axes[i].patches:
            axes[i].annotate(
                f'{int(p.get_height()):,}',
                (p.get_x() + p.get_width() / 2., p.get_height()),
                ha='center', va='bottom', fontsize=9
            )
    
    plt.tight_layout()
    plt.show()
else:
    print("No categorical features found.")

In [ ]:
# Claim rate by categorical features
if n_cat > 0:
    fig, axes = plt.subplots(1, n_cat, figsize=(6 * n_cat, 5))
    if n_cat == 1:
        axes = [axes]
    
    for i, col in enumerate(categorical_cols):
        claim_rate = df.groupby(col)[target_col].mean().sort_values(ascending=False)
        bars = axes[i].bar(
            claim_rate.index, claim_rate.values,
            color=plt.cm.RdYlGn_r(claim_rate.values / claim_rate.values.max()),
            edgecolor='black', alpha=0.85
        )
        axes[i].set_title(f'Claim Rate by {col}', fontsize=13, fontweight='bold')
        axes[i].set_ylabel('Claim Rate')
        axes[i].tick_params(axis='x', rotation=45)
        axes[i].axhline(df[target_col].mean(), color='red', linestyle='--', alpha=0.7,
                        label=f'Overall: {df[target_col].mean():.3f}')
        axes[i].legend(fontsize=10)
        
        # Annotate percentages
        for bar, rate in zip(bars, claim_rate.values):
            axes[i].text(
                bar.get_x() + bar.get_width()/2., bar.get_height() + 0.005,
                f'{rate:.1%}', ha='center', va='bottom', fontsize=10, fontweight='bold'
            )
    
    plt.tight_layout()
    plt.show()

    # Print claim rates table
    print("\nClaim Rates by Category:")
    for col in categorical_cols:
        print(f"\n  {col}:")
        rates = df.groupby(col)[target_col].agg(['mean', 'count'])
        rates.columns = ['Claim Rate', 'Count']
        rates = rates.sort_values('Claim Rate', ascending=False)
        for idx, row in rates.iterrows():
            print(f"    {idx:15s} -> {row['Claim Rate']:.3f}  (n={int(row['Count']):,})")

In [ ]:
# Stacked bar chart: categorical features by claim status
if n_cat > 0:
    fig, axes = plt.subplots(1, n_cat, figsize=(6 * n_cat, 5))
    if n_cat == 1:
        axes = [axes]
    
    for i, col in enumerate(categorical_cols):
        ct = pd.crosstab(df[col], df[target_col], normalize='index')
        ct.plot(kind='bar', stacked=True, ax=axes[i],
                color=['#3498db', '#e74c3c'], edgecolor='black', alpha=0.85)
        axes[i].set_title(f'{col} - Claim Proportion', fontsize=13, fontweight='bold')
        axes[i].set_ylabel('Proportion')
        axes[i].legend(['No Claim', 'Claim'], loc='upper right')
        axes[i].tick_params(axis='x', rotation=45)
    
    plt.tight_layout()
    plt.show()

## 9. Feature Engineering

Creating new features to improve model performance:
- **Interaction features**: Combining existing features to capture joint effects
- **Binning**: Converting continuous variables into categories
- **Encoding**: Transforming categorical variables for model consumption

In [ ]:
# Work on a copy of the dataframe
df_eng = df.copy()

print("Original features:", df_eng.shape[1])
print()

# --- Interaction Features ---
print("=" * 50)
print("Creating Interaction Features")
print("=" * 50)

# Premium per age: higher premium relative to age might indicate higher risk
if 'annual_premium' in df_eng.columns and 'age' in df_eng.columns:
    df_eng['premium_per_age'] = df_eng['annual_premium'] / (df_eng['age'] + 1)
    print("  Created: premium_per_age = annual_premium / (age + 1)")

# Claims per tenure: claim frequency relative to policy duration
if 'num_claims_hist' in df_eng.columns and 'policy_tenure' in df_eng.columns:
    df_eng['claims_per_tenure'] = df_eng['num_claims_hist'] / (df_eng['policy_tenure'] + 0.1)
    print("  Created: claims_per_tenure = num_claims_hist / (policy_tenure + 0.1)")

# Premium to credit ratio: financial risk indicator
if 'annual_premium' in df_eng.columns and 'credit_score' in df_eng.columns:
    df_eng['premium_credit_ratio'] = df_eng['annual_premium'] / (df_eng['credit_score'] + 1)
    print("  Created: premium_credit_ratio = annual_premium / (credit_score + 1)")

# Risk score: composite risk indicator
if all(c in df_eng.columns for c in ['num_claims_hist', 'credit_score', 'age']):
    df_eng['risk_score'] = (
        df_eng['num_claims_hist'] * 100
        + (850 - df_eng['credit_score'])
        + np.where(df_eng['age'] < 25, 50, 0)
        + np.where(df_eng['age'] > 65, 30, 0)
    )
    print("  Created: risk_score (composite of claims, credit, age)")

In [ ]:
# --- Binning ---
print("=" * 50)
print("Creating Binned Features")
print("=" * 50)

# Age groups
if 'age' in df_eng.columns:
    df_eng['age_group'] = pd.cut(
        df_eng['age'],
        bins=[0, 25, 35, 45, 55, 65, 100],
        labels=['18-25', '26-35', '36-45', '46-55', '56-65', '65+']
    )
    print("  Created: age_group (6 bins)")
    print(f"    {df_eng['age_group'].value_counts().to_dict()}")

# Credit score tiers
if 'credit_score' in df_eng.columns:
    df_eng['credit_tier'] = pd.cut(
        df_eng['credit_score'],
        bins=[0, 580, 670, 740, 800, 850],
        labels=['Poor', 'Fair', 'Good', 'Very Good', 'Excellent']
    )
    print("  Created: credit_tier (5 tiers)")

# Premium quartiles
if 'annual_premium' in df_eng.columns:
    df_eng['premium_quartile'] = pd.qcut(
        df_eng['annual_premium'], q=4, labels=['Q1', 'Q2', 'Q3', 'Q4'],
        duplicates='drop'
    )
    print("  Created: premium_quartile (4 quartiles)")

# Log transform for skewed features
if 'annual_premium' in df_eng.columns:
    df_eng['log_premium'] = np.log1p(df_eng['annual_premium'])
    print("  Created: log_premium = log(1 + annual_premium)")

In [ ]:
# --- Encoding ---
print("=" * 50)
print("Encoding Categorical Features")
print("=" * 50)

# Vehicle age numeric mapping
if 'vehicle_age' in df_eng.columns and df_eng['vehicle_age'].dtype == 'object':
    vehicle_age_map = {'< 1 Year': 0, '1-2 Year': 1, '> 2 Years': 2}
    df_eng['vehicle_age_numeric'] = df_eng['vehicle_age'].map(vehicle_age_map).fillna(1)
    print(f"  Encoded: vehicle_age -> vehicle_age_numeric {vehicle_age_map}")

# Label encoding for other categoricals
label_encoders = {}
cat_cols_to_encode = df_eng.select_dtypes(include=['object', 'category']).columns.tolist()

for col in cat_cols_to_encode:
    le = LabelEncoder()
    df_eng[f'{col}_encoded'] = le.fit_transform(df_eng[col].astype(str))
    label_encoders[col] = le
    print(f"  Label encoded: {col} -> {col}_encoded")
    mapping = dict(zip(le.classes_, le.transform(le.classes_)))
    print(f"    Mapping: {mapping}")

print(f"\nFinal engineered dataset shape: {df_eng.shape}")
print(f"New features created: {df_eng.shape[1] - df.shape[1]}")

In [ ]:
# Visualize claim rates across engineered bins
binned_features = [c for c in ['age_group', 'credit_tier', 'premium_quartile'] if c in df_eng.columns]

if binned_features:
    fig, axes = plt.subplots(1, len(binned_features), figsize=(6 * len(binned_features), 5))
    if len(binned_features) == 1:
        axes = [axes]
    
    for i, col in enumerate(binned_features):
        claim_rate = df_eng.groupby(col)[target_col].mean()
        bars = axes[i].bar(
            range(len(claim_rate)), claim_rate.values,
            color=plt.cm.RdYlGn_r(claim_rate.values / max(claim_rate.values.max(), 0.01)),
            edgecolor='black', alpha=0.85
        )
        axes[i].set_xticks(range(len(claim_rate)))
        axes[i].set_xticklabels(claim_rate.index, rotation=45, ha='right')
        axes[i].set_title(f'Claim Rate by {col}', fontsize=13, fontweight='bold')
        axes[i].set_ylabel('Claim Rate')
        axes[i].axhline(df_eng[target_col].mean(), color='red', linestyle='--', alpha=0.7)
        
        for bar, rate in zip(bars, claim_rate.values):
            axes[i].text(
                bar.get_x() + bar.get_width()/2., bar.get_height() + 0.003,
                f'{rate:.1%}', ha='center', fontsize=10, fontweight='bold'
            )
    
    plt.suptitle('Claim Rate Across Engineered Features', fontsize=15, fontweight='bold')
    plt.tight_layout()
    plt.show()

In [ ]:
# Updated correlation heatmap with new features
new_numeric_cols = df_eng.select_dtypes(include=[np.number]).columns.tolist()
corr_with_target = df_eng[new_numeric_cols].corr()[target_col].drop(target_col).abs().sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(10, max(6, len(corr_with_target) * 0.4)))
colors = plt.cm.RdYlGn_r(corr_with_target.values / max(corr_with_target.values.max(), 0.01))
bars = ax.barh(range(len(corr_with_target)), corr_with_target.values, color=colors, edgecolor='black', alpha=0.85)
ax.set_yticks(range(len(corr_with_target)))
ax.set_yticklabels(corr_with_target.index)
ax.set_xlabel('Absolute Correlation with Target', fontsize=12)
ax.set_title('Feature Importance (Correlation with Claim Filed)', fontsize=14, fontweight='bold')

for bar, val in zip(bars, corr_with_target.values):
    ax.text(bar.get_width() + 0.002, bar.get_y() + bar.get_height()/2.,
            f'{val:.4f}', va='center', fontsize=9)

plt.tight_layout()
plt.show()

## 10. Summary of Findings

### Key Observations

**Dataset Characteristics:**
- The dataset contains policyholder demographics, vehicle information, and historical claims data
- Target variable (`claim_filed`) is binary with some class imbalance - models should use class weighting or resampling

**Feature Insights:**
- **Age**: Younger policyholders (18-25) show higher claim rates, consistent with actuarial data
- **Vehicle Age**: Older vehicles (`> 2 Years`) correlate with higher claim probability
- **Credit Score**: Lower credit scores associate with higher claim rates (below 600 is a strong indicator)
- **Historical Claims**: `num_claims_hist` is one of the strongest predictors - past behavior predicts future claims
- **Annual Premium**: Higher premiums may reflect higher risk profiles

**Feature Engineering Results:**
- `premium_per_age` captures the interaction between premium and age, normalizing premium by policyholder age
- `claims_per_tenure` provides a claim frequency measure normalized by policy duration
- `risk_score` composite feature combines multiple risk factors into a single metric
- Binned features (`age_group`, `credit_tier`) create more interpretable groupings

**Next Steps:**
- Train multiple classification models (Logistic Regression, Random Forest, XGBoost, LightGBM)
- Apply probability calibration for reliable risk scores
- Optimize decision thresholds for business value
- Use SHAP for model interpretability

Proceed to **Notebook 02** for model training, evaluation, and explainability analysis.